# Regression - Data Modeling

Use the cleaned Chicago crime dataset to prepare model features, run baseline experiments, and inspect results that can guide a regression-oriented modeling strategy.

In [1]:
from pathlib import Path
import sys

import pandas as pd
from sklearn.metrics import accuracy_score, classification_report

PROJECT_ROOT = Path.cwd().resolve().parents[1] if Path.cwd().name == 'regression' else Path.cwd().resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.modeling import (
    EXPERIMENT_LOG,
    ensure_modeling_columns,
    prepare_features,
    run_experiments,
    run_hourbin2_models,
)
from src.preprocess import load_cleaned

CLEANED_PATH = PROJECT_ROOT / 'data' / 'processed' / 'crimes_cleaned.csv'


In [2]:
df = load_cleaned(CLEANED_PATH)
df = ensure_modeling_columns(df, sample_size=100000)
print('Modeling shape:', df.shape)
df.head()

Modeling shape: (100000, 29)


,ID,Date,Block,IUCR,Primary Type,Description,Location Description,Arrest,Domestic,Beat,...,DayOfWeekName,Hour,Quarter,WeekOfYear,IsWeekend,Lat_bin,Lon_bin,HourBin,Crime_Category,Location_enc
1049005,12549853,2021-11-23 18:00:00,021XX S INDIANA AVE,0820,THEFT,$500 AND UNDER,STREET,0,0,132,...,Tuesday,18,4,47,False,2092,-4382,4,Theft,19
888931,12804930,2022-08-18 00:00:00,019XX W 87TH ST,2250,LIQUOR LAW VIOLATION,LIQUOR LICENSE VIOLATION,SMALL RETAIL STORE,1,0,2221,...,Thursday,0,3,33,False,2086,-4384,0,Other,18
1336428,12096981,2020-07-03 22:00:00,006XX N CICERO AVE,0460,BATTERY,SIMPLE,RESIDENCE,0,0,1111,...,Friday,22,3,27,False,2094,-4388,5,Violent,12
797182,12935007,2022-12-27 14:50:00,035XX S MORGAN ST,0810,THEFT,OVER $500,COMMERCIAL / BUSINESS OFFICE,0,0,915,...,Tuesday,14,4,52,False,2091,-4383,3,Theft,3
1024032,12597206,2022-01-12 00:00:00,105XX S LA SALLE ST,0610,BURGLARY,FORCIBLE ENTRY,RESIDENCE,0,0,512,...,Wednesday,0,1,2,False,2085,-4382,0,Theft,12


In [3]:
prepared = prepare_features(df)
prepared.keys()

dict_keys(['X1', 'y1', 'X2', 'y2', 'X3_alt', 'y3_2bin', 'feat_crime', 'feat_beat2', 'feat_hour_alt'])

In [4]:
EXPERIMENT_LOG.clear()

crime_model, X1_test, y1_test, _ = run_experiments(prepared['X1'], prepared['y1'], 'Crime_Category')
beat_model, X2_test, y2_test, _ = run_experiments(prepared['X2'], prepared['y2'], 'Beat')
X3_test, y3_test, y3_pred = run_hourbin2_models(prepared['X3_alt'], prepared['y3_2bin'])

/Users/ashmil/Documents/PhD in Information Technology/DataScienceAndBigDataAnalysis/chicago-crime-analysis/.venv/lib/python3.11/site-packages/joblib/externals/loky/process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


In [5]:
experiment_results = pd.DataFrame(EXPERIMENT_LOG).sort_values(['task', 'accuracy'], ascending=[True, False])
experiment_results

,task,model,params,accuracy,f1_weighted,notes
6,Beat,GB,{},0.8461,0.8438,
7,Beat,GB_tuned,"{'learning_rate': 0.05, 'max_depth': 5, 'n_est...",0.8443,0.8421,GridSearch
5,Beat,RF,{},0.8244,0.8228,
4,Beat,LR,{},0.8153,0.8034,
3,Crime_Category,GB_tuned,"{'learning_rate': 0.05, 'max_depth': 5, 'n_est...",0.5074,0.4582,GridSearch
2,Crime_Category,GB,{},0.5042,0.4474,
1,Crime_Category,RF,{},0.4570,0.4347,
0,Crime_Category,LR,{},0.4521,0.3658,
8,HourBin2,GB,{},0.5979,0.4729,2 bins
10,HourBin2,SVM_linear,{},0.5969,0.4463,2 bins


In [6]:
print('HourBin2 accuracy:', round(accuracy_score(y3_test, y3_pred), 4))
print(classification_report(y3_test, y3_pred))

HourBin2 accuracy: 0.5979
              precision    recall  f1-score   support

           0       0.52      0.04      0.07      8061
           1       0.60      0.98      0.74     11939

    accuracy                           0.60     20000
   macro avg       0.56      0.51      0.41     20000
weighted avg       0.57      0.60      0.47     20000



## Next Regression Step

For a true regression notebook, aggregate the cleaned events into count targets such as incidents per beat-hour, community-month, or location-day, then train regressors like Random Forest Regressor, Gradient Boosting Regressor, XGBoost, or Poisson/Negative Binomial models.